In [1]:
! pip install torch

In [2]:
import torch
import torch.nn as nn
import math

In [3]:
# Example sentence
sentence = ["ChatGPT", "is", "great", "at", "explaining", "things", "clearly"]
seq_len = len(sentence)
vocab_size = 10000
embedding_dim = 16
num_heads = 2
hidden_dim = 32

In [4]:
# 1️⃣ Token embedding
token_embedding = nn.Embedding(vocab_size, embedding_dim)
token_ids = torch.arange(seq_len)
tokens_emb = token_embedding(token_ids)  # [seq_len, embedding_dim]
print("Token embeddings:\n", tokens_emb, "\n")

Token embeddings:
 tensor([[ 4.8541e-01,  3.2486e-01, -2.4489e-01,  1.3199e+00, -1.6304e+00,
          1.0798e+00,  3.4318e-01, -7.8734e-01, -1.7832e-01,  1.0872e+00,
         -6.5937e-01,  9.4759e-01,  1.4896e+00,  4.6761e-01, -2.5932e-01,
          1.7052e+00],
        [-4.8788e-01,  9.7018e-01,  3.3032e-01,  6.9756e-01,  1.8503e+00,
          7.5048e-03,  1.7710e+00, -5.6246e-01,  2.1183e-02, -4.1724e-01,
         -5.8002e-01,  2.3392e+00, -1.0003e+00, -2.8931e-01, -1.3301e-02,
          7.6880e-01],
        [ 2.2001e-01, -1.2725e-01,  2.1717e-01,  1.6576e+00,  1.7661e+00,
         -9.9336e-01,  2.1654e+00,  9.5546e-01,  1.0242e+00,  2.8664e-01,
          3.2213e+00, -1.1209e+00,  7.9113e-01,  7.3562e-01,  2.1554e-01,
          4.1880e-01],
        [ 2.8059e+00,  4.0835e-01, -4.3241e-01,  2.7008e-01, -1.4055e+00,
         -9.3041e-01,  1.4982e+00, -3.3817e-01,  2.1131e+00, -9.9899e-01,
         -1.4330e-01,  8.7683e-01, -1.3191e+00,  1.2369e+00,  8.6553e-01,
          3.1107e-01],
 

In [5]:
# 2️⃣ Positional encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

In [6]:
pos_encoder = PositionalEncoding(embedding_dim)
emb_with_pos = pos_encoder(tokens_emb)  # [seq_len, embedding_dim]
print("Embeddings + positional encoding:\n", emb_with_pos)

Embeddings + positional encoding:
 tensor([[ 4.8541e-01,  1.3249e+00, -2.4489e-01,  2.3199e+00, -1.6304e+00,
          2.0798e+00,  3.4318e-01,  2.1266e-01, -1.7832e-01,  2.0872e+00,
         -6.5937e-01,  1.9476e+00,  1.4896e+00,  1.4676e+00, -2.5932e-01,
          2.7052e+00],
        [ 3.5359e-01,  1.5105e+00,  6.4130e-01,  1.6480e+00,  1.9501e+00,
          1.0025e+00,  1.8026e+00,  4.3704e-01,  3.1182e-02,  5.8271e-01,
         -5.7686e-01,  3.3392e+00, -9.9926e-01,  7.1069e-01, -1.2985e-02,
          1.7688e+00],
        [ 1.1293e+00, -5.4339e-01,  8.0829e-01,  2.4642e+00,  1.9648e+00,
         -1.3297e-02,  2.2286e+00,  1.9535e+00,  1.0442e+00,  1.2864e+00,
          3.2276e+00, -1.2089e-01,  7.9313e-01,  1.7356e+00,  2.1618e-01,
          1.4188e+00],
        [ 2.9470e+00, -5.8164e-01,  3.8024e-01,  8.5284e-01, -1.1100e+00,
          2.4926e-02,  1.5929e+00,  6.5734e-01,  2.1431e+00,  5.6291e-04,
         -1.3381e-01,  1.8768e+00, -1.3161e+00,  2.2368e+00,  8.6648e-01,
        

In [7]:
# 3️⃣ Tiny Transformer Encoder layer
encoder_layer = nn.TransformerEncoderLayer(
    d_model=embedding_dim,
    nhead=num_heads,
    dim_feedforward=hidden_dim
)
transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)

/Users/wick/language_models/.venv/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [8]:
# Transformer expects [seq_len, batch_size, embedding_dim]
# Here batch_size=1
emb_with_pos = emb_with_pos.unsqueeze(1)
output = transformer_encoder(emb_with_pos)

print("Transformer output shape:", output.shape)
print("Transformer output:\n", output.squeeze(1))

Transformer output shape: torch.Size([7, 1, 16])
Transformer output:
 tensor([[ 0.0616,  0.5893, -0.9318,  1.1894, -1.8406,  0.6467, -0.7743, -0.5609,
         -0.4603,  0.8497, -1.2592,  0.9090,  1.1232,  0.0279, -1.1895,  1.6195],
        [-0.7573,  0.6493, -0.2946,  0.4843,  1.7991,  0.7118, -0.1007, -0.4094,
         -0.5246, -0.8579, -0.9885,  2.2010, -1.1367, -0.8963, -0.8954,  1.0160],
        [-0.2477, -1.4938, -0.6163,  1.2668,  0.9922, -1.2837,  0.0384,  1.2147,
          0.0851, -0.5193,  1.9239, -1.1317,  0.1166,  0.5099, -1.3535,  0.4984],
        [ 1.9095, -1.1742, -0.5449,  0.1568, -1.4603, -0.5995,  0.1614,  0.3032,
          1.4843, -0.9445, -0.8133,  1.1511, -1.2565,  1.0652,  0.0678,  0.4938],
        [-0.1991,  0.2769,  1.0774, -0.3174, -2.1018,  1.3732, -0.6408,  2.3142,
         -1.2216, -0.8214, -0.1978,  0.0601, -0.3479,  0.2961,  0.3530,  0.0967],
        [ 0.8335, -0.4945, -0.0257,  0.1223,  0.0880, -0.2028, -2.3824,  0.1008,
          1.6122,  1.3342, -1.2633